# Montly GDP with Stock and Watson (2010) Methods

In [57]:
import pandas as pd
from pathlib import Path
import os
import time
import numpy as np
from functools import reduce
import csv
from typing import Union
import akshare as ak

In [5]:
os.chdir('/Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data')
print("Current Directory:", os.getcwd())

Current Directory: /Users/liminglin/我的云端硬盘 (liminglin1998@gmail.com)/Sciences Po/Thesis/Data


## Consumption

In [48]:
consump_df = pd.read_csv(
    "Social Consumption Retail Sales.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)
consump_df = consump_df.iloc[:2, :].copy()

consump_df= consump_df.T.rename_axis("date").reset_index()

# 2) clean date text like "2026年2月" -> "2026-02"
consump_df["date"] = pd.to_datetime(consump_df["date"], format="%Y年%m月", errors="coerce")

consump_df.columns = ["date", "current consumption", "cumulative consumption"]

# Feb cumulative by year
year = consump_df["date"].dt.year
month = consump_df["date"].dt.month
feb_cum_by_year = consump_df.loc[month.eq(2)].set_index(year[month.eq(2)])["cumulative consumption"]

# fill Jan and Feb current = (Feb cumulative / 2)
for m in [1, 2]:
    mask = month.eq(m) & consump_df["current consumption"].isna()   # only fill missing
    consump_df.loc[mask, "current consumption"] = year[mask].map(feb_cum_by_year) / 2

## Trade

In [ ]:
trade_df = pd.read_csv(
    "Export-Import.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)

trade_df = trade_df.iloc[[12], :].copy()

trade_df = trade_df.T.rename_axis("date").reset_index()
# 2) clean date text like "2026年2月" -> "2026-02"
trade_df["date"] = pd.to_datetime(trade_df["date"], format="%Y年%m月", errors="coerce")

trade_df.columns = ["date", "trade balance"]

## Fixed Asset Investment

In [ ]:
fai_df = pd.read_csv(
    "Fixed Asset Investment.csv",
    encoding="gbk",
    sep=",",
    header=2,      # use the 3rd row as the column names (dates)
    index_col=0,   # use the first column as the row labels (variables)
    engine="python",
)


In [55]:
fai_df = fai_df.iloc[[0], :].copy()

In [56]:
fai_df = fai_df.T.rename_axis("date").reset_index()
# 2) clean date text like "2026年2月" -> "2026-02"
fai_df["date"] = pd.to_datetime(fai_df["date"], format="%Y年%m月", errors="coerce")

fai_df.columns = ["date", "fai growth"]

In [58]:
urbanfai_df = ak.macro_china_gdzctz()